# Emirates NBD IFRS S1/S2 Style Extraction — Azure OpenAI Debug Notebook

This notebook extracts style patterns from the Emirates NBD IFRS S1/S2 report using **Azure OpenAI** instead of AzureOpenAI/Azure OpenAI.

**Workflow**
1. Configure paths and Azure OpenAI credentials
2. Extract PDF pages with `pdfplumber`
3. Classify pages into IFRS S1/S2 report sections
4. Build section chunks
5. Analyze section style with Azure OpenAI
6. Synthesize a unified style guide
7. Build reusable LangGraph agent prompts
8. Save `emirates_nbd_style_reference.json`

In [ ]:
# Optional installs
# Run this only if the packages are missing in your environment.
# !pip install pdfplumber openai python-dotenv tqdm

In [1]:
# ── Imports and logging ─────────────────────────────────────
import os
import re
import json
import logging
from pathlib import Path
from typing import Optional

import pdfplumber
from dotenv import load_dotenv
from tqdm import tqdm

try:
    from openai import AzureOpenAI
except ImportError:
    AzureOpenAI = None

load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
log = logging.getLogger(__name__)

print("Imports ready")
print("Azure OpenAI package available:", AzureOpenAI is not None)

Imports ready
Azure OpenAI package available: True


In [ ]:
# ── Configuration ───────────────────────────────────────────
# This cell now searches more intelligently for the Emirates NBD PDF.
#
# Best options:
# 1. Put the PDF in the same folder as this notebook, or in a nearby data/ or reports/ folder.
# 2. Or set EMIRATES_NBD_PDF_PATH in your .env.
# 3. Or manually set PDF_PATH_MANUAL below.

# Manual override example:
# PDF_PATH_MANUAL = Path(r"C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\emirates_nbd_group_2024_ifrs_s1_s2.pdf")
PDF_PATH_MANUAL = None

OUTPUT_PATH = Path("emirates_nbd_style_reference.json")
DEBUG = True

def _normalise_pdf_path(value):
    if value is None:
        return None
    p = Path(str(value)).expanduser()
    return p if p.exists() and p.suffix.lower() == ".pdf" else None

def find_emirates_pdf() -> Path | None:
    """
    Find the Emirates NBD IFRS S1/S2 PDF from:
    - manual override
    - .env variable EMIRATES_NBD_PDF_PATH
    - common nearby paths
    - recursive search in current project folders
    """
    # 1) Manual override
    manual = _normalise_pdf_path(PDF_PATH_MANUAL)
    if manual:
        print("Using manual PDF path:", manual)
        return manual

    # 2) Environment variable
    env_path = _normalise_pdf_path(os.getenv("EMIRATES_NBD_PDF_PATH"))
    if env_path:
        print("Using EMIRATES_NBD_PDF_PATH:", env_path)
        return env_path

    # 3) Exact common paths
    exact_names = [
        "emirates_nbd_group_2024_ifrs_s1_s2.pdf",
        "emirates_nbd_2024_ifrs_s1_s2.pdf",
        "Emirates NBD Group 2024 IFRS S1 S2.pdf",
        "Emirates_NBD_Group_2024_IFRS_S1_S2.pdf",
        "emirates_nbd_group_2024_sustainability_report.pdf",
    ]

    roots = [
        Path.cwd(),
        Path.cwd() / "data",
        Path.cwd() / "reports",
        Path.cwd() / "reference_reports",
        Path.cwd() / "gen_data",
        Path.cwd() / "gen_data" / "reports",
        Path.cwd() / "gen_data" / "reference_reports",
        Path.cwd().parent,
        Path.cwd().parent / "data",
        Path.cwd().parent / "reports",
        Path.cwd().parent / "reference_reports",
        Path.cwd().parent / "gen_data",
        Path.cwd().parent / "gen_data" / "reports",
        Path.cwd().parent / "gen_data" / "reference_reports",
        Path("/mnt/data"),
    ]

    checked_exact = []
    for root in roots:
        for name in exact_names:
            candidate = root / name
            checked_exact.append(candidate)
            if candidate.exists() and candidate.suffix.lower() == ".pdf":
                print("Found PDF:", candidate)
                return candidate

    # 4) Recursive search by patterns
    patterns = [
        "*emirates*nbd*.pdf",
        "*Emirates*NBD*.pdf",
        "*ifrs*s1*s2*.pdf",
        "*IFRS*S1*S2*.pdf",
    ]

    recursive_roots = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/mnt/data"),
    ]

    matches = []
    for root in recursive_roots:
        if not root.exists():
            continue
        for pattern in patterns:
            try:
                matches.extend(root.rglob(pattern))
            except Exception as exc:
                print(f"Could not recursively search {root}: {exc}")

    # De-duplicate and prefer Emirates NBD names
    unique_matches = []
    seen = set()
    for p in matches:
        key = str(p.resolve()) if p.exists() else str(p)
        if key not in seen and p.is_file() and p.suffix.lower() == ".pdf":
            seen.add(key)
            unique_matches.append(p)

    if unique_matches:
        unique_matches = sorted(
            unique_matches,
            key=lambda p: (
                0 if "emirates" in p.name.lower() and "nbd" in p.name.lower() else 1,
                len(str(p))
            )
        )
        print("Found PDF by recursive search:", unique_matches[0])
        print("Other PDF matches:", [str(p) for p in unique_matches[1:5]])
        return unique_matches[0]

    print("No Emirates NBD PDF found automatically.")
    print("Checked common exact paths such as:")
    for p in checked_exact[:12]:
        print(" -", p)
    print("Set PDF_PATH_MANUAL in this cell or add EMIRATES_NBD_PDF_PATH to your .env.")
    return None

PDF_PATH = find_emirates_pdf()

# ── Azure OpenAI settings ───────────────────────────────────
# Recommended .env variables:
# AZURE_OPENAI_API_KEY=...
# AZURE_OPENAI_ENDPOINT=https://<your-resource>.openai.azure.com/
# AZURE_OPENAI_API_VERSION=2024-10-21
# AZURE_OPENAI_DEPLOYMENT=<your-chat-deployment-name>
#
# Optional alternative env names are also supported:
# AZURE_OPENAI_CHAT_DEPLOYMENT, AZURE_OPENAI_MODEL, AZURE_OPENAI_DEPLOYMENT_NAME

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21")
AZURE_OPENAI_DEPLOYMENT = (
    os.getenv("AZURE_OPENAI_DEPLOYMENT")
    or os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
    or os.getenv("AZURE_OPENAI_MODEL")
    or os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
)

# Manual Azure override examples:
# AZURE_OPENAI_ENDPOINT = "https://your-resource.openai.azure.com/"
# AZURE_OPENAI_API_KEY = "paste-your-key-here"   # Not recommended to hard-code in shared notebooks
# AZURE_OPENAI_DEPLOYMENT = "gpt-4.1"            # Your Azure deployment name, not necessarily the base model name
# AZURE_OPENAI_API_VERSION = "2024-10-21"

print("Current working directory:", Path.cwd())
print("PDF_PATH:", PDF_PATH)
print("OUTPUT_PATH:", OUTPUT_PATH.resolve())
print("DEBUG:", DEBUG)
print("Azure endpoint configured:", bool(AZURE_OPENAI_ENDPOINT))
print("Azure key configured:", bool(AZURE_OPENAI_API_KEY))
print("Azure deployment:", AZURE_OPENAI_DEPLOYMENT)
print("Azure API version:", AZURE_OPENAI_API_VERSION)

In [4]:
# ── Section detection config ────────────────────────────────
# Maps IFRS S1/S2 pillar names to keywords that identify them in the PDF text.
# Adjust these if the report uses different heading labels.

SECTION_KEYWORDS = {
    "governance": [
        "governance", "board oversight", "board of directors",
        "management role", "committee", "oversight"
    ],
    "strategy": [
        "strategy", "strategic", "climate-related risks", "opportunities",
        "scenario analysis", "transition plan", "resilience",
        "time horizon", "physical risk", "transition risk"
    ],
    "risk_management": [
        "risk management", "risk identification", "risk assessment",
        "enterprise risk", "climate risk integration", "risk appetite",
        "risk framework"
    ],
    "metrics_and_targets": [
        "metrics", "targets", "scope 1", "scope 2", "scope 3",
        "ghg emissions", "carbon", "tco2e", "net zero",
        "financed emissions", "baseline year", "emissions intensity"
    ]
}

SKIP_PAGE_PATTERNS = [
    r"^table of contents",
    r"^contents$",
    r"^appendix",
    r"^\d+$",
]

MAX_CHUNK_WORDS = 1200

print("Section keywords ready")

Section keywords ready


In [5]:
# ── Step 1: PDF page extraction ─────────────────────────────
def extract_pdf_pages(pdf_path: str | Path, debug: bool = False) -> list[dict]:
    '''
    Extract text from each page of the PDF.
    Returns list of {page_num, text, word_count} dicts.
    '''
    pdf_path = Path(pdf_path)
    pages = []
    log.info(f"Opening PDF: {pdf_path}")

    with pdfplumber.open(pdf_path) as pdf:
        total = len(pdf.pages)
        log.info(f"Total pages: {total}")

        for i, page in enumerate(tqdm(pdf.pages, desc="Extracting pages")):
            text = page.extract_text() or ""
            text = text.strip()

            if not text or len(text) < 80:
                continue

            first_line = text.split("\n")[0].lower().strip()
            if any(re.match(p, first_line) for p in SKIP_PAGE_PATTERNS):
                if debug:
                    log.debug(f"Skipping page {i+1}: matched skip pattern")
                continue

            pages.append({
                "page_num": i + 1,
                "text": text,
                "word_count": len(text.split())
            })

    log.info(f"Extracted {len(pages)} content pages (skipped {total - len(pages)})")
    return pages


print("extract_pdf_pages() ready")

extract_pdf_pages() ready


In [ ]:
# ── Run Step 1: Extract pages ───────────────────────────────
if PDF_PATH is None:
    raise FileNotFoundError(
        "PDF_PATH is None. The notebook could not find the Emirates NBD PDF automatically.\n\n"
        "Fix it in the Configuration cell by setting:\n"
        "PDF_PATH_MANUAL = Path(r'C:\\path\\to\\emirates_nbd_group_2024_ifrs_s1_s2.pdf')\n\n"
        "Or add this to your .env file:\n"
        "EMIRATES_NBD_PDF_PATH=C:\\path\\to\\emirates_nbd_group_2024_ifrs_s1_s2.pdf\n\n"
        "Then rerun the Configuration cell and this cell."
    )

pages = extract_pdf_pages(PDF_PATH, debug=DEBUG)

print("Pages extracted:", len(pages))
print("First 5 extracted pages:")
for page in pages[:5]:
    print(f"- page {page['page_num']} | words={page['word_count']} | first line={page['text'].splitlines()[0][:100]}")

In [ ]:
# ── Step 2: Page classification ─────────────────────────────
def classify_page_section(page_text: str) -> Optional[str]:
    '''
    Assign a page to one of the 4 IFRS S1/S2 pillars based on keyword density.
    Returns the section name or None if unclassified.
    '''
    text_lower = page_text.lower()
    scores = {}

    for section, keywords in SECTION_KEYWORDS.items():
        score = sum(text_lower.count(kw) for kw in keywords)
        if score > 0:
            scores[section] = score

    if not scores:
        return None

    return max(scores, key=scores.get)


def classify_page_scores(page_text: str) -> dict:
    '''
    Debug helper: return all keyword scores, not only the winning section.
    '''
    text_lower = page_text.lower()
    return {
        section: sum(text_lower.count(kw) for kw in keywords)
        for section, keywords in SECTION_KEYWORDS.items()
    }


def group_pages_by_section(pages: list[dict]) -> dict[str, list[dict]]:
    '''
    Group extracted pages by IFRS section.
    Returns {section_name: [page_dicts]}
    '''
    sections: dict[str, list[dict]] = {
        "governance": [],
        "strategy": [],
        "risk_management": [],
        "metrics_and_targets": [],
        "general": []
    }

    for page in pages:
        section = classify_page_section(page["text"])
        if section:
            sections[section].append(page)
        else:
            sections["general"].append(page)

    for name, pages_list in sections.items():
        log.info(f"  {name}: {len(pages_list)} pages")

    return sections


print("Classification functions ready")

In [ ]:
# ── Run Step 2: Group pages by section ──────────────────────
section_pages = group_pages_by_section(pages)

print("\nSection page counts:")
for section, pg_list in section_pages.items():
    print(f"- {section}: {len(pg_list)} pages")

print("\nPage classification debug sample:")
for page in pages[:10]:
    scores = classify_page_scores(page["text"])
    winner = classify_page_section(page["text"])
    print(f"page {page['page_num']:>3} -> {winner or 'general'} | scores={scores}")

In [ ]:
# ── Step 3: Build section chunks ────────────────────────────
def build_section_chunk(pages: list[dict], max_words: int = MAX_CHUNK_WORDS) -> str:
    '''
    Concatenate pages for a section up to max_words.
    Returns a single string chunk.
    '''
    combined = []
    total = 0

    for page in pages:
        words = page["text"].split()
        if total + len(words) > max_words:
            remaining = max_words - total
            combined.append(" ".join(words[:remaining]))
            break
        combined.append(page["text"])
        total += len(words)

    return "\n\n".join(combined)


print("build_section_chunk() ready")

In [ ]:
# ── Run Step 3: Build chunks ────────────────────────────────
section_chunks = {}

for section, pg_list in section_pages.items():
    if pg_list:
        chunk = build_section_chunk(pg_list, max_words=MAX_CHUNK_WORDS)
        section_chunks[section] = chunk
        print(f"{section}: {len(chunk.split())} words")

print("\nChunk preview:")
for section, chunk in section_chunks.items():
    print("\n" + "="*80)
    print(section.upper())
    print("="*80)
    print(chunk[:900])

In [ ]:
# ── Step 4: Azure OpenAI section analysis prompts ─────────────────
SECTION_PROMPTS = {
    "governance": """
You are a sustainability reporting analyst. Analyze this GOVERNANCE section from an IFRS S1/S2 bank sustainability report.

Extract the following and return ONLY valid JSON — no preamble, no markdown fences:

{
  "section": "governance",
  "tone_descriptors": ["list of 4-6 adjectives describing the writing tone"],
  "sentence_structure": "short paragraph describing typical sentence length and complexity",
  "example_sentences": ["3 verbatim sentences that best represent the reporting style"],
  "opening_patterns": ["2-3 ways this section typically opens a paragraph"],
  "board_language": ["exact phrases used to describe board/committee oversight roles"],
  "formality_markers": ["list of formal phrases/constructions used, e.g. 'The Board has established...'"],
  "tense_usage": "present/past/mixed — and how each is used",
  "passive_vs_active": "ratio estimate and pattern description",
  "hedging_phrases": ["phrases used to soften or qualify statements"],
  "section_length_estimate": "estimated word count range for this section"
}

REPORT TEXT:
{text}
""",

    "strategy": """
You are a sustainability reporting analyst. Analyze this STRATEGY section from an IFRS S1/S2 bank sustainability report.

Extract the following and return ONLY valid JSON — no preamble, no markdown fences:

{
  "section": "strategy",
  "tone_descriptors": ["list of 4-6 adjectives describing the writing tone"],
  "sentence_structure": "short paragraph describing typical sentence length and complexity",
  "example_sentences": ["3 verbatim sentences that best represent the reporting style"],
  "risk_language": {
    "physical_risk_phrases": ["exact phrases used to describe physical climate risks"],
    "transition_risk_phrases": ["exact phrases used to describe transition risks"],
    "opportunity_phrases": ["phrases used to describe climate opportunities"]
  },
  "time_horizon_language": ["how short/medium/long-term horizons are expressed in text"],
  "scenario_analysis_phrases": ["exact phrases used when referencing scenario analysis"],
  "forward_looking_language": ["hedged phrases for forward-looking statements, e.g. 'The Group expects...'"],
  "financial_impact_language": ["how financial impacts are described without specific numbers"],
  "formality_markers": ["key formal phrases or constructions"],
  "section_length_estimate": "estimated word count range for this section"
}

REPORT TEXT:
{text}
""",

    "risk_management": """
You are a sustainability reporting analyst. Analyze this RISK MANAGEMENT section from an IFRS S1/S2 bank sustainability report.

Extract the following and return ONLY valid JSON — no preamble, no markdown fences:

{
  "section": "risk_management",
  "tone_descriptors": ["list of 4-6 adjectives describing the writing tone"],
  "sentence_structure": "short paragraph describing typical sentence length and complexity",
  "example_sentences": ["3 verbatim sentences that best represent the reporting style"],
  "process_description_language": ["phrases used to describe identification/assessment processes"],
  "integration_language": ["how climate risk integration into ERM is expressed"],
  "risk_classification_language": ["how risk categories or tiers are described"],
  "governance_link_phrases": ["phrases that connect risk management back to governance"],
  "formality_markers": ["key formal phrases or constructions"],
  "hedging_phrases": ["phrases used to qualify risk statements"],
  "section_length_estimate": "estimated word count range for this section"
}

REPORT TEXT:
{text}
""",

    "metrics_and_targets": """
You are a sustainability reporting analyst. Analyze this METRICS AND TARGETS section from an IFRS S1/S2 bank sustainability report.

Extract the following and return ONLY valid JSON — no preamble, no markdown fences:

{
  "section": "metrics_and_targets",
  "tone_descriptors": ["list of 4-6 adjectives describing the writing tone"],
  "sentence_structure": "short paragraph describing typical sentence length and complexity",
  "example_sentences": ["3 verbatim sentences that best represent the reporting style"],
  "emissions_reporting_style": {
    "scope_1_phrasing": "how scope 1 is introduced and described",
    "scope_2_phrasing": "how scope 2 is introduced and described",
    "scope_3_phrasing": "how scope 3 is introduced and described",
    "financed_emissions_phrasing": "how financed emissions are described if present"
  },
  "target_description_pattern": "how targets are structured in text (baseline → progress → goal)",
  "unit_presentation": ["how units like tCO2e, MWh are presented inline"],
  "methodology_citation_style": "how calculation methodologies are referenced",
  "table_introduction_phrases": ["phrases used to introduce data tables"],
  "year_on_year_comparison_language": ["phrases used for YoY comparisons"],
  "formality_markers": ["key formal phrases or constructions"],
  "section_length_estimate": "estimated word count range for this section"
}

REPORT TEXT:
{text}
""",

    "general": """
You are a sustainability reporting analyst. Analyze this text from an IFRS S1/S2 bank sustainability report.

Extract general style patterns and return ONLY valid JSON — no preamble, no markdown fences:

{
  "section": "general",
  "report_voice": "first-person plural / third-person / mixed — with description",
  "organization_reference_style": "how the bank refers to itself (e.g. 'the Group', 'Emirates NBD', 'we')",
  "paragraph_length": "typical paragraph word count range",
  "list_usage": "how bullet points and numbered lists are used — sparingly/frequently/never",
  "cross_reference_style": "how other sections or documents are cross-referenced",
  "boilerplate_phrases": ["5-8 recurring formal phrases found throughout the report"],
  "disclosure_disclaimer_language": ["phrases used for regulatory disclaimer statements"],
  "year_reference_style": "how reporting year is referred to (e.g. '2024', 'the reporting period', 'the year under review')",
  "document_title_style": "how section and subsection titles are formatted"
}

REPORT TEXT:
{text}
"""
}

print("SECTION_PROMPTS ready:", list(SECTION_PROMPTS.keys()))

In [ ]:
# ── Step 5: Azure OpenAI API helper ─────────────────────────
def parse_json_response(raw: str) -> dict:
    '''
    Strip accidental markdown fences and parse JSON.
    '''
    raw = raw.strip()
    raw = re.sub(r"^```json\s*", "", raw)
    raw = re.sub(r"^```\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)
    return json.loads(raw)


def get_azure_openai_client() -> AzureOpenAI:
    '''
    Build Azure OpenAI client from environment variables or manual config cell values.
    '''
    if AzureOpenAI is None:
        raise ImportError("openai package is not installed. Run the install cell first.")

    missing = []
    if not AZURE_OPENAI_API_KEY:
        missing.append("AZURE_OPENAI_API_KEY")
    if not AZURE_OPENAI_ENDPOINT:
        missing.append("AZURE_OPENAI_ENDPOINT")
    if not AZURE_OPENAI_DEPLOYMENT:
        missing.append("AZURE_OPENAI_DEPLOYMENT or AZURE_OPENAI_CHAT_DEPLOYMENT")

    if missing:
        raise EnvironmentError(
            "Missing Azure OpenAI settings: " + ", ".join(missing) +
            ". Set them in your .env file or in the configuration cell."
        )

    return AzureOpenAI(
        api_key=AZURE_OPENAI_API_KEY,
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
        api_version=AZURE_OPENAI_API_VERSION,
    )


def azure_chat_json(
    client: AzureOpenAI,
    system_prompt: str,
    user_prompt: str,
    max_tokens: int = 2000,
    temperature: float = 0,
) -> dict:
    '''
    Calls Azure OpenAI and returns parsed JSON.
    Uses JSON response format when supported; falls back to normal text JSON if needed.
    '''
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    try:
        response = client.chat.completions.create(
            model=AZURE_OPENAI_DEPLOYMENT,
            messages=messages,
            max_tokens=max_tokens,
            temperature=temperature,
            response_format={"type": "json_object"},
        )
    except Exception as first_error:
        # Some Azure deployments/API versions may not support response_format.
        log.warning(f"Retrying without response_format because first call failed: {first_error}")
        response = client.chat.completions.create(
            model=AZURE_OPENAI_DEPLOYMENT,
            messages=messages,
            max_tokens=max_tokens,
            temperature=temperature,
        )

    raw = response.choices[0].message.content.strip()
    return parse_json_response(raw)


def analyze_section_with_azure(
    client: AzureOpenAI,
    section_name: str,
    text_chunk: str,
    debug: bool = False
) -> Optional[dict]:
    '''
    Send a section chunk to Azure OpenAI for style analysis.
    Returns parsed JSON dict or None on failure.
    '''
    if not text_chunk.strip():
        log.warning(f"Empty chunk for section '{section_name}' — skipping")
        return None

    prompt_template = SECTION_PROMPTS.get(section_name, SECTION_PROMPTS["general"])
    prompt = prompt_template.format(text=text_chunk)

    if debug:
        log.info(f"Sending {len(text_chunk.split())} words to Azure OpenAI for '{section_name}'")

    try:
        parsed = azure_chat_json(
            client=client,
            system_prompt=(
                "You are a document style analyst. "
                "You always respond with valid JSON only — no markdown, no preamble, no explanation. "
                "If you cannot extract a field, use null. Never invent data not present in the text."
            ),
            user_prompt=prompt,
            max_tokens=2000,
            temperature=0,
        )

        log.info(f"  ✓ '{section_name}' analysis complete")
        return parsed

    except json.JSONDecodeError as e:
        log.error(f"JSON parse error for '{section_name}': {e}")
        return None
    except Exception as e:
        log.error(f"Azure OpenAI API error for '{section_name}': {e}")
        return None


print("Azure OpenAI helper functions ready")

In [ ]:
# ── Run Step 5: Analyze one section first for debugging ─────
# Start with one section before running all sections.
# Change TEST_SECTION if needed.

TEST_SECTION = "governance"

client = get_azure_openai_client()

test_analysis = analyze_section_with_azure(
    client=client,
    section_name=TEST_SECTION,
    text_chunk=section_chunks.get(TEST_SECTION, ""),
    debug=DEBUG,
)

print(json.dumps(test_analysis, indent=2, ensure_ascii=False) if test_analysis else "No analysis returned")

In [ ]:
# ── Run Step 5b: Analyze all sections ───────────────────────
# Run after the test section works.

section_analyses = {}

for section, chunk in section_chunks.items():
    print("\n" + "="*80)
    print("Analyzing:", section)
    print("="*80)

    analysis = analyze_section_with_azure(
        client=client,
        section_name=section,
        text_chunk=chunk,
        debug=DEBUG,
    )
    section_analyses[section] = analysis

print("\nCompleted analyses:")
for section, analysis in section_analyses.items():
    print(f"- {section}: {'OK' if analysis else 'FAILED'}")

In [ ]:
# ── Step 6: Synthesis prompt ────────────────────────────────
SYNTHESIS_PROMPT = """
You are a document style consultant. You have analyzed sections of an IFRS S1/S2 sustainability report.

Here are the per-section style analyses:
{section_analyses}

Synthesize these into a unified style guide for use by AI agents generating new report sections.

Return ONLY valid JSON — no preamble, no markdown fences:

{
  "report_identity": {
    "organization_name": "how the bank refers to itself in text",
    "reporting_year": "year or period reference style used",
    "document_type": "type of document style (formal corporate report, etc.)"
  },
  "universal_style_rules": [
    "8-10 rules that apply across ALL sections, written as actionable instructions for an AI writer"
  ],
  "tone_profile": {
    "primary_tone": "single best descriptor",
    "secondary_tones": ["2-3 secondary descriptors"],
    "what_to_avoid": ["3-5 tone/style anti-patterns NOT present in this report"]
  },
  "section_specific_instructions": {
    "governance": "2-3 sentence writing instruction for governance section agent",
    "strategy": "2-3 sentence writing instruction for strategy section agent",
    "risk_management": "2-3 sentence writing instruction for risk management section agent",
    "metrics_and_targets": "2-3 sentence writing instruction for metrics section agent"
  },
  "common_opening_patterns": ["5 ways sections or paragraphs typically begin"],
  "common_closing_patterns": ["3 ways sections or paragraphs typically end"],
  "master_phrase_bank": ["20-25 exact phrases or sentence starters from the report to reuse in generation"],
  "formatting_conventions": {
    "heading_style": "description of heading formatting",
    "table_style": "description of how tables are presented",
    "list_style": "description of bullet/numbered list usage",
    "number_formatting": "how numbers and percentages are written"
  }
}
"""

print("SYNTHESIS_PROMPT ready")

In [ ]:
# ── Step 6: Synthesize unified style guide ──────────────────
def synthesize_style_guide(client: AzureOpenAI, section_analyses: dict) -> Optional[dict]:
    '''
    Call Azure OpenAI to synthesize all section analyses into a unified style guide.
    '''
    log.info("Synthesizing unified style guide...")

    valid_analyses = {k: v for k, v in section_analyses.items() if v is not None}

    if not valid_analyses:
        log.error("No valid section analyses to synthesize")
        return None

    prompt = SYNTHESIS_PROMPT.format(
        section_analyses=json.dumps(valid_analyses, indent=2, ensure_ascii=False)
    )

    try:
        parsed = azure_chat_json(
            client=client,
            system_prompt=(
                "You are a document style consultant. "
                "Respond with valid JSON only. No markdown, no preamble."
            ),
            user_prompt=prompt,
            max_tokens=3000,
            temperature=0,
        )

        log.info("  ✓ Synthesis complete")
        return parsed

    except json.JSONDecodeError as e:
        log.error(f"Synthesis JSON parse error: {e}")
        return None
    except Exception as e:
        log.error(f"Synthesis Azure OpenAI API error: {e}")
        return None


style_guide = synthesize_style_guide(client, section_analyses)

print(json.dumps(style_guide, indent=2, ensure_ascii=False) if style_guide else "No style guide generated")

In [ ]:
# ── Step 7: Build LangGraph agent prompts ───────────────────
def build_agent_prompts(style_guide: dict) -> dict[str, str]:
    '''
    Build system prompts for each LangGraph section agent using extracted style patterns.
    '''
    if not style_guide:
        return {}

    universal_rules = "\n".join(
        f"- {rule}"
        for rule in style_guide.get("universal_style_rules", [])
    )

    phrase_bank = "\n".join(
        f'- "{phrase}"'
        for phrase in style_guide.get("master_phrase_bank", [])
    )

    org_name = style_guide.get("report_identity", {}).get("organization_name", "the Group")
    primary_tone = style_guide.get("tone_profile", {}).get("primary_tone", "formal")

    prompts = {}

    for section in ["governance", "strategy", "risk_management", "metrics_and_targets"]:
        section_instruction = (
            style_guide
            .get("section_specific_instructions", {})
            .get(section, "Write in a formal, professional tone.")
        )

        avoid = "\n".join(
            f"- {item}"
            for item in style_guide.get("tone_profile", {}).get("what_to_avoid", [])
        )

        prompts[section] = f'''You are an IFRS S1/S2 sustainability reporting specialist writing for {org_name}.

SECTION: {section.upper().replace("_", " ")}

STYLE INSTRUCTION:
{section_instruction}

UNIVERSAL STYLE RULES:
{universal_rules}

TONE: {primary_tone}

AVOID:
{avoid}

PHRASE BANK:
{phrase_bank}

TASK:
Given the JSON input data for the {section.replace("_", " ")} section, write the complete disclosure narrative.

Rules:
- Do not invent data not present in the input JSON.
- If a required disclosure is missing, state the evidence boundary in report-style wording.
- Match the style patterns above.
- Output the section text only.
'''

    return prompts


agent_prompts = build_agent_prompts(style_guide)

print("Agent prompts generated:", len(agent_prompts))
for section, prompt in agent_prompts.items():
    print("\n" + "="*80)
    print(section.upper())
    print("="*80)
    print(prompt[:1200])

In [ ]:
# ── Step 8: Assemble and save output ────────────────────────
output = {
    "metadata": {
        "source_pdf": str(Path(PDF_PATH).name) if PDF_PATH else None,
        "pages_extracted": len(pages),
        "sections_found": {k: len(v) for k, v in section_pages.items()},
        "extraction_model": "AZURE_OPENAI_DEPLOYMENT",
    },
    "per_section_analysis": section_analyses,
    "unified_style_guide": style_guide,
    "agent_prompts": agent_prompts,
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print("Saved output to:", OUTPUT_PATH.resolve())
print("Sections analyzed:", sum(1 for v in section_analyses.values() if v), "/", len(section_analyses))
print("Agent prompts:", list(agent_prompts.keys()))

In [ ]:
# ── Optional: Save each agent prompt as a separate .txt file ─
prompt_dir = Path("emirates_style_agent_prompts")
prompt_dir.mkdir(exist_ok=True)

for section, prompt in agent_prompts.items():
    path = prompt_dir / f"{section}_style_prompt.txt"
    path.write_text(prompt, encoding="utf-8")
    print("Saved:", path)

In [ ]:
# ── Optional: Reload and inspect final JSON ─────────────────
with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    ref = json.load(f)

print("Top-level keys:", list(ref.keys()))
print("Metadata:")
print(json.dumps(ref.get("metadata", {}), indent=2, ensure_ascii=False))

print("\nUniversal style rules:")
for i, rule in enumerate(ref.get("unified_style_guide", {}).get("universal_style_rules", []), 1):
    print(f"{i}. {rule}")